# EEG Analysis — OpenNeuro ds001787 (Meditation)
**Main library:** MNE-Python  
**Pipeline:** BIDS loading → Preprocessing → Flat channel detection → Artifact rejection → PSD analysis

> 💡 Every cell has detailed comments. Run them top to bottom in order.

---
## 0. Install dependencies
Run this cell only once.

In [ ]:
# Install required libraries (only needed once)
# !pip install mne mne-bids openneuro-py matplotlib numpy scipy

---
## 1. Import libraries

In [ ]:
import mne
import mne_bids
import openneuro
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Suppress less important log messages
mne.set_log_level('WARNING')

print(f"MNE version: {mne.__version__}")
print(f"MNE-BIDS version: {mne_bids.__version__}")

---
## 2. Download dataset from OpenNeuro

Dataset ds001787 is an EEG study on **meditation** in BIDS format.  
The folder structure will be:
```
ds001787/
├── participants.tsv       ← subject info
├── dataset_description.json
└── sub-01/
    └── eeg/
        ├── sub-01_task-meditation_eeg.set   ← EEG data
        ├── sub-01_task-meditation_channels.tsv
        └── sub-01_task-meditation_events.tsv
```

In [ ]:
# Directory where the data will be saved
bids_root = Path('./ds001787')
bids_root.mkdir(exist_ok=True)

# Download only the first subject to start
# (remove include=[...] to download the full dataset)
openneuro.download(
    dataset='ds001787',
    target_dir=bids_root,
    include=['sub-01']  # subject 1 only
)

print("Download complete!")

---
## 3. Explore dataset structure

In [ ]:
# Print folder structure (up to 3 levels deep)
mne_bids.print_dir_tree(bids_root, max_depth=3)

In [ ]:
import pandas as pd

# Read participant information
participants_file = bids_root / 'participants.tsv'
if participants_file.exists():
    df = pd.read_csv(participants_file, sep='\t')
    print(f"Number of subjects: {len(df)}")
    display(df.head())
else:
    print("participants.tsv not found")

---
## 4. Load EEG data with MNE-BIDS

In [ ]:
# Create a BIDSPath — the object MNE uses to locate files
# Adjust subject and task_name based on the actual dataset
subject = '01'
task_name = 'meditation'  # <-- may differ, verify with print_dir_tree above

bids_path = mne_bids.BIDSPath(
    subject=subject,
    task=task_name,
    root=bids_root,
    datatype='eeg'
)

print(f"File path: {bids_path.fpath}")

# Load raw data
raw = mne_bids.read_raw_bids(bids_path, verbose=False)

print("\n--- RECORDING INFO ---")
print(f"Sampling frequency: {raw.info['sfreq']} Hz")
print(f"Number of EEG channels: {len(raw.ch_names)}")
print(f"Duration: {raw.times[-1]:.1f} s ({raw.times[-1]/60:.1f} min)")
print(f"Channels: {raw.ch_names}")

---
## 5. Preprocessing

Preprocessing removes noise from the EEG signal before analysis.  
Steps applied here:
1. **Band-pass filter 1–40 Hz** → keeps only the frequencies of interest
2. **Average reference** → redistributes the signal across all electrodes

In [ ]:
# Copy raw data so the original stays untouched
raw_preprocessed = raw.copy()

# --- STEP 1: Load data into memory ---
raw_preprocessed.load_data()

# --- STEP 2: Band-pass filter 1–40 Hz ---
# - 1 Hz high-pass: removes slow drift (movement, sweat, cable sway)
# - 40 Hz low-pass: removes high-frequency noise (muscles, electronics)
# Note: the 40 Hz low-pass already suppresses any 50 Hz line noise,
#       so a separate notch filter is not needed here.
raw_preprocessed.filter(
    l_freq=1.0,    # high-pass cutoff
    h_freq=40.0,   # low-pass cutoff
    method='fir',
    verbose=False
)
print("✅ Band-pass filter 1–40 Hz applied")

# --- STEP 3: Average re-reference ---
# EEG is always relative to a reference electrode.
# Using the average of all channels is a common, robust choice.
raw_preprocessed.set_eeg_reference('average', projection=False, verbose=False)
print("✅ Average reference applied")

print("\n🎉 Preprocessing complete!")

In [ ]:
# Plot raw vs preprocessed signal for comparison (first 10 s, first 5 channels)
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

n_channels_to_plot = min(5, len(raw.ch_names))
t_start, t_end = 0, 10  # seconds
sfreq = raw.info['sfreq']

start_idx = int(t_start * sfreq)
end_idx = int(t_end * sfreq)
times = raw.times[start_idx:end_idx]

# Raw signal
data_raw, _ = raw[:n_channels_to_plot, start_idx:end_idx]
for i, ch in enumerate(raw.ch_names[:n_channels_to_plot]):
    axes[0].plot(times, data_raw[i] * 1e6 + i * 100, label=ch, lw=0.8)
axes[0].set_title('RAW signal', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Amplitude (µV)')
axes[0].legend(loc='upper right', fontsize=7)

# Preprocessed signal
data_prep, _ = raw_preprocessed[:n_channels_to_plot, start_idx:end_idx]
for i, ch in enumerate(raw_preprocessed.ch_names[:n_channels_to_plot]):
    axes[1].plot(times, data_prep[i] * 1e6 + i * 100, label=ch, lw=0.8)
axes[1].set_title('PREPROCESSED signal (band-pass 1–40 Hz)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Amplitude (µV)')
axes[1].set_xlabel('Time (s)')
axes[1].legend(loc='upper right', fontsize=7)

plt.tight_layout()
plt.savefig('raw_vs_preprocessed.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'raw_vs_preprocessed.png'")

---
## 6. Flat channel detection

A **flat channel** is an electrode that records little or no signal — it is either disconnected,
has dried gel, or has a hardware fault. Including flat channels contaminates the average reference
and inflates z-score thresholds in the next step, so we identify and exclude them first.

**Detection criteria (both must be true to flag a channel as flat):**
- Standard deviation of the signal < `FLAT_STD_THRESHOLD` (µV)
- Peak-to-peak amplitude < `FLAT_PTP_THRESHOLD` (µV)

In [ ]:
# ─────────────────────────────────────────────
# PARAMETERS — adjust if needed
# ─────────────────────────────────────────────
FLAT_STD_THRESHOLD = 0.5e-6   # std < 0.5 µV  → likely flat
FLAT_PTP_THRESHOLD = 1.0e-6   # ptp < 1.0 µV  → likely flat
# ─────────────────────────────────────────────

data_check, _ = raw_preprocessed[:, :]

ch_std = data_check.std(axis=1)   # std for each channel across the entire recording
ch_ptp = np.ptp(data_check, axis=1)  # peak-to-peak for each channel

# A channel is flagged as flat if BOTH criteria are met
flat_mask = (ch_std < FLAT_STD_THRESHOLD) & (ch_ptp < FLAT_PTP_THRESHOLD)
flat_channels = [raw_preprocessed.ch_names[i] for i in np.where(flat_mask)[0]]

print(f"Flat channel detection — thresholds: std < {FLAT_STD_THRESHOLD*1e6:.1f} µV, ptp < {FLAT_PTP_THRESHOLD*1e6:.1f} µV")
print(f"Total channels checked: {len(raw_preprocessed.ch_names)}")
print(f"Flat channels found:    {len(flat_channels)}")
if flat_channels:
    print(f"  → {flat_channels}")
else:
    print("  → None detected ✅")

In [ ]:
# Visualise std and ptp for all channels — flat ones highlighted in red
import pandas as pd

df_ch = pd.DataFrame({
    'channel': raw_preprocessed.ch_names,
    'std_uV':  ch_std * 1e6,
    'ptp_uV':  ch_ptp * 1e6,
    'flat':    flat_mask
})

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors = ['#e74c3c' if f else '#3498db' for f in df_ch['flat']]

axes[0].bar(df_ch['channel'], df_ch['std_uV'], color=colors)
axes[0].axhline(FLAT_STD_THRESHOLD * 1e6, color='black', lw=1.5,
                linestyle='--', label=f'Threshold ({FLAT_STD_THRESHOLD*1e6:.1f} µV)')
axes[0].set_title('Per-channel standard deviation', fontsize=11)
axes[0].set_ylabel('Std (µV)')
axes[0].set_xticklabels(df_ch['channel'], rotation=90, fontsize=6)
axes[0].legend()

axes[1].bar(df_ch['channel'], df_ch['ptp_uV'], color=colors)
axes[1].axhline(FLAT_PTP_THRESHOLD * 1e6, color='black', lw=1.5,
                linestyle='--', label=f'Threshold ({FLAT_PTP_THRESHOLD*1e6:.1f} µV)')
axes[1].set_title('Per-channel peak-to-peak amplitude', fontsize=11)
axes[1].set_ylabel('Peak-to-peak (µV)')
axes[1].set_xticklabels(df_ch['channel'], rotation=90, fontsize=6)
axes[1].legend()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='Flat'), Patch(facecolor='#3498db', label='Good')]
fig.legend(handles=legend_elements, loc='upper right', fontsize=9)

plt.suptitle('Flat channel detection', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('flat_channel_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'flat_channel_detection.png'")

In [ ]:
# Mark flat channels as bads in the MNE info object
# This is the standard MNE approach: channels are excluded from computations
# (e.g. average reference, PSD) but are NOT physically removed from the data,
# so you can always inspect or recover them later.
if flat_channels:
    # Merge with any bads already set (e.g. from the BIDS sidecar)
    existing_bads = raw_preprocessed.info['bads']
    new_bads = list(set(existing_bads + flat_channels))
    raw_preprocessed.info['bads'] = new_bads
    print(f"Marked {len(flat_channels)} flat channel(s) as bads: {flat_channels}")
    print(f"All bads in info: {raw_preprocessed.info['bads']}")
else:
    print("No flat channels found — info['bads'] unchanged ✅")

# Verify: MNE will now automatically exclude bads from the average reference
# and from PSD computations unless you explicitly pass picks='all'
print(f"\nTotal channels: {len(raw_preprocessed.ch_names)}  |  Bads: {len(raw_preprocessed.info['bads'])}  |  Good: {len(raw_preprocessed.ch_names) - len(raw_preprocessed.info['bads'])}")

---
## 7. Artifact rejection with Z-score on 2-second epochs

### What is an EEG artifact?
An **artifact** is a spurious signal that does not originate from brain activity:  
eye movements (EOG), muscle contractions (EMG), cable movement, heartbeat, etc.

### Strategy: per-channel Z-score across epochs
1. Split the continuous signal into **2-second epochs**
2. For each epoch and each channel, compute the **peak-to-peak amplitude** (max − min)
3. Compute the **z-score** of that amplitude *across all epochs of the same channel*
4. If the z-score exceeds a **threshold** (default: 3.0), the epoch is flagged as artifacted

```
z = (epoch_peak - mean_of_peaks) / std_of_peaks
```
> A z-score > 3 means the peak is more than 3 standard deviations above the mean → very likely an artifact.

In [ ]:
# ─────────────────────────────────────────────
# PARAMETERS — adjust if needed
# ─────────────────────────────────────────────
EPOCH_DURATION = 2.0   # epoch length in seconds
Z_THRESHOLD    = 3.0   # z-score threshold (3.0 = conservative, 2.5 = stricter)
# ─────────────────────────────────────────────

sfreq = raw_preprocessed.info['sfreq']
epoch_samples = int(EPOCH_DURATION * sfreq)  # samples per epoch

# Get EEG data as a numpy array: shape (n_channels, n_samples)
data, times_raw = raw_preprocessed[:, :]
n_channels, n_samples = data.shape

# How many complete epochs fit in the recording?
n_epochs = n_samples // epoch_samples
print(f"Total samples:              {n_samples}")
print(f"Samples per epoch ({EPOCH_DURATION}s):  {epoch_samples}")
print(f"Number of complete epochs:  {n_epochs}")

In [ ]:
# ── STEP 1: compute peak-to-peak amplitude per epoch and channel ──
# peak-to-peak = max(signal) - min(signal) within the epoch
# resulting shape: (n_channels, n_epochs)

peak_to_peak = np.zeros((n_channels, n_epochs))

for ep in range(n_epochs):
    start = ep * epoch_samples
    end   = start + epoch_samples
    epoch_data = data[:, start:end]                         # shape: (n_channels, epoch_samples)
    peak_to_peak[:, ep] = epoch_data.max(axis=1) - epoch_data.min(axis=1)

print(f"Peak-to-peak computed → shape: {peak_to_peak.shape}  (channels × epochs)")

In [ ]:
# ── STEP 2: compute z-score per channel ──
# For each channel: z = (value - mean) / std
# The z-score is computed ACROSS epochs (axis=1), not across channels

mean_ptp = peak_to_peak.mean(axis=1, keepdims=True)   # shape: (n_channels, 1)
std_ptp  = peak_to_peak.std(axis=1, keepdims=True)    # shape: (n_channels, 1)

# Avoid division by zero for perfectly flat channels (already removed, but safe)
std_ptp[std_ptp == 0] = 1e-10

z_scores = (peak_to_peak - mean_ptp) / std_ptp        # shape: (n_channels, n_epochs)

print(f"Z-scores computed → shape: {z_scores.shape}")
print(f"Global max z-score: {z_scores.max():.2f}")
print(f"Global min z-score: {z_scores.min():.2f}")

In [ ]:
# ── STEP 3: identify epochs to reject ──
# An epoch is rejected if AT LEAST ONE channel exceeds the threshold

# Boolean mask: True = artifact in that channel/epoch combination
artifact_mask = z_scores > Z_THRESHOLD          # shape: (n_channels, n_epochs)

# Epochs with at least 1 artifacted channel
epochs_to_reject = np.any(artifact_mask, axis=0)  # shape: (n_epochs,)
epochs_clean     = ~epochs_to_reject

n_rejected = epochs_to_reject.sum()
n_clean    = epochs_clean.sum()
pct_rejected = 100 * n_rejected / n_epochs

print(f"\n{'='*45}")
print(f"Z-score threshold:       {Z_THRESHOLD}")
print(f"Total epochs:            {n_epochs}")
print(f"REJECTED epochs:         {n_rejected}  ({pct_rejected:.1f}%)")
print(f"CLEAN epochs:            {n_clean}  ({100-pct_rejected:.1f}%)")
print(f"{'='*45}")

if pct_rejected > 40:
    print("⚠️  More than 40% rejected — consider raising the threshold (e.g. 3.5 or 4.0)")
elif pct_rejected < 5:
    print("ℹ️  Less than 5% rejected — conservative threshold, looks good")
else:
    print("✅  Rejection rate within normal range")

In [ ]:
# ── STEP 4: visualise rejected epochs ──
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# --- Subplot 1: z-score heatmap (channels × epochs) ---
epoch_indices = np.arange(n_epochs)
im = axes[0].imshow(
    z_scores,
    aspect='auto',
    cmap='RdYlGn_r',       # red = high z-score, green = low
    vmin=0, vmax=Z_THRESHOLD * 1.5,
    interpolation='nearest'
)
plt.colorbar(im, ax=axes[0], label='Z-score')
axes[0].set_xlabel('Epoch (2 s)')
axes[0].set_ylabel('Channel')
axes[0].set_title(f'Z-score heatmap — channels × epochs  |  threshold = {Z_THRESHOLD}', fontsize=11)

max_z_per_epoch = z_scores.max(axis=0)

# --- Subplot 2: max z-score per epoch with threshold line ---
axes[1].bar(epoch_indices[epochs_clean],
            max_z_per_epoch[epochs_clean],
            color='#2ecc71', alpha=0.7, width=1.0, label='Clean epoch')
axes[1].bar(epoch_indices[epochs_to_reject],
            max_z_per_epoch[epochs_to_reject],
            color='#e74c3c', alpha=0.85, width=1.0, label='Rejected epoch')
axes[1].axhline(y=Z_THRESHOLD, color='black', lw=1.5,
                linestyle='--', label=f'Threshold z={Z_THRESHOLD}')
axes[1].set_xlabel('Epoch (2 s)', fontsize=11)
axes[1].set_ylabel('Max z-score', fontsize=11)
axes[1].set_title(f'Max z-score per epoch  |  {n_rejected}/{n_epochs} rejected ({pct_rejected:.1f}%)', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('artifact_rejection_zscore.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'artifact_rejection_zscore.png'")

In [ ]:
# ── STEP 5: reconstruct the clean signal ──
# Concatenate only the clean epochs → continuous signal without artifacts

clean_segments = []
for ep in range(n_epochs):
    if epochs_clean[ep]:
        start = ep * epoch_samples
        end   = start + epoch_samples
        clean_segments.append(data[:, start:end])

# Concatenate all clean segments
data_clean = np.concatenate(clean_segments, axis=1)  # shape: (n_channels, n_clean_samples)

duration_orig  = n_samples / sfreq
duration_clean = data_clean.shape[1] / sfreq

print(f"Original duration:     {duration_orig:.1f} s  ({duration_orig/60:.1f} min)")
print(f"Duration after cleaning: {duration_clean:.1f} s  ({duration_clean/60:.1f} min)")
print(f"Signal retained:       {100*duration_clean/duration_orig:.1f}%")

# Create a new MNE RawArray object with the clean data
raw_clean = mne.io.RawArray(data_clean, raw_preprocessed.info, verbose=False)
print("\n✅ raw_clean created — ready for PSD!")

In [ ]:
# ── Per-channel artifact summary: how many epochs flagged per channel? ──
import pandas as pd

artifacts_per_channel = artifact_mask.sum(axis=1)  # n. of artifacted epochs per channel
pct_per_channel = 100 * artifacts_per_channel / n_epochs

df_artifacts = pd.DataFrame({
    'Channel': raw_preprocessed.ch_names,
    'Artifacted epochs': artifacts_per_channel,
    '% of total': pct_per_channel.round(1)
}).sort_values('Artifacted epochs', ascending=False)

print("\n--- TOP 10 noisiest channels ---")
display(df_artifacts.head(10))

# Flag channels with more than 30% of epochs artifacted
bad_channels = df_artifacts[df_artifacts['% of total'] > 30]['Channel'].tolist()
if bad_channels:
    print(f"\n⚠️  Channels with >30% artifacted epochs (consider excluding them):")
    print(f"   {bad_channels}")
else:
    print("\n✅ No excessively noisy channels (>30%)")

---
## 8. Frequency analysis — PSD (Power Spectral Density)

The PSD measures **how much energy** is present at each frequency in the EEG signal.  

> 🧹 From this point we use **`raw_clean`** — the signal already cleaned of artifacts and flat channels.

Main **EEG frequency bands**:

| Band | Range (Hz) | Associated with |
|------|-----------|----------------|
| Delta | 0.5 – 4 | Deep sleep |
| **Theta** | **4 – 8** | **Meditation, creativity** |
| **Alpha** | **8 – 13** | **Relaxation, meditation** |
| Beta | 13 – 30 | Active thinking, attention |
| Gamma | 30 – 40 | Higher cognitive processes |

> In this meditation dataset we expect a prominent **alpha peak (~10 Hz)**.

In [ ]:
# Compute PSD using Welch's method
# (splits the signal into overlapping segments and averages their spectra)
psd = raw_clean.compute_psd(
    method='welch',
    fmin=1.0,    # lowest frequency to compute
    fmax=40.0,   # highest frequency to compute
    n_fft=2048,  # frequency resolution (higher = more detail)
    n_overlap=512
)

print("PSD computed!")
print(f"PSD data shape: {psd.get_data().shape}  (channels × frequencies)")
print(f"Frequency resolution: {psd.freqs[1] - psd.freqs[0]:.3f} Hz")

In [ ]:
# PSD plot — average across all channels
fig, ax = plt.subplots(figsize=(12, 5))

# Get data and frequencies
psd_data = psd.get_data()  # shape: (n_channels, n_freqs)
freqs = psd.freqs

# Convert to dB: 10 * log10(power)
psd_db = 10 * np.log10(psd_data)

# Mean and standard deviation across channels
psd_mean = psd_db.mean(axis=0)
psd_std = psd_db.std(axis=0)

# Mean line
ax.plot(freqs, psd_mean, color='#2f4a8c', lw=2, label='Mean across channels')

# Uncertainty band (±1 std)
ax.fill_between(freqs, psd_mean - psd_std, psd_mean + psd_std,
                alpha=0.2, color='#2f4a8c', label='±1 SD')

# Colour frequency bands
bands = {
    'Delta\n(0.5–4)': (0.5, 4, '#e8d5f5'),
    'Theta\n(4–8)': (4, 8, '#d5e8f5'),
    'Alpha\n(8–13)': (8, 13, '#d5f5e3'),
    'Beta\n(13–30)': (13, 30, '#f5f0d5'),
    'Gamma\n(30–40)': (30, 40, '#f5d5d5'),
}

for band_name, (f_lo, f_hi, color) in bands.items():
    ax.axvspan(f_lo, f_hi, alpha=0.3, color=color)
    ax.text((f_lo + f_hi) / 2, ax.get_ylim()[0] if ax.get_ylim()[0] > -100 else psd_mean.min() - 2,
            band_name, ha='center', va='bottom', fontsize=8, color='#555')

ax.set_xlabel('Frequency (Hz)', fontsize=12)
ax.set_ylabel('Power (dB)', fontsize=12)
ax.set_title('Power Spectral Density — Mean across all channels\nds001787 (Meditation)', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(1, 40)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('psd_global.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'psd_global.png'")

In [ ]:
# Compute relative power for each frequency band
# (useful for cross-subject and cross-condition comparisons)

def band_power(psd_data, freqs, fmin, fmax):
    """Return mean power in the band [fmin, fmax] Hz."""
    idx = np.logical_and(freqs >= fmin, freqs <= fmax)
    return psd_data[:, idx].mean(axis=1)  # mean over frequencies, per channel

band_definitions = {
    'Delta (0.5–4 Hz)':   (0.5, 4),
    'Theta (4–8 Hz)':     (4, 8),
    'Alpha (8–13 Hz)':    (8, 13),
    'Beta (13–30 Hz)':    (13, 30),
    'Gamma (30–40 Hz)':   (30, 40),
}

print("--- BAND POWER (mean across all channels) ---\n")
results = {}
for band_name, (fmin, fmax) in band_definitions.items():
    power = band_power(psd_data, freqs, fmin, fmax)
    mean_power_db = 10 * np.log10(power.mean())
    results[band_name] = mean_power_db
    print(f"{band_name:25s}: {mean_power_db:.2f} dB")

In [ ]:
# Band power bar chart
fig, ax = plt.subplots(figsize=(8, 5))

colors = ['#9b59b6', '#3498db', '#2ecc71', '#f39c12', '#e74c3c']
bands_short = ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']
values = list(results.values())

bars = ax.bar(bands_short, values, color=colors, edgecolor='white', linewidth=1.5)

# Add value labels above bars
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Mean power (dB)', fontsize=12)
ax.set_title('Band power comparison\nds001787 — sub-01', fontsize=12)
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(min(values) - 5, max(values) + 5)

plt.tight_layout()
plt.savefig('band_power_barplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'band_power_barplot.png'")

---
## 9. PSD before vs after processing

This section computes the PSD on the **raw signal** and compares it side-by-side with
the **clean signal** (`raw_clean`), so you can visually assess the impact of the full
preprocessing + artifact rejection pipeline.

- **Left subplot** — raw signal PSD (no filtering, no artifact removal)
- **Right subplot** — clean signal PSD (band-pass 1–40 Hz + bads excluded + z-score rejection)

In [ ]:
# ── Compute PSD on the original raw signal (before any processing) ──
# We load it fresh so we are comparing truly unprocessed data
raw_for_comparison = raw.copy().load_data()

psd_raw = raw_for_comparison.compute_psd(
    method='welch',
    fmin=0.5,
    fmax=80.0,   # wider range to show line noise and muscle artifacts if present
    n_fft=2048,
    n_overlap=512
)

# ── PSD on the clean signal (already computed as `psd` in section 8) ──
# We recompute with the same fmin/fmax for a fair comparison
psd_clean = raw_clean.compute_psd(
    method='welch',
    fmin=0.5,
    fmax=80.0,
    n_fft=2048,
    n_overlap=512
)

print("PSDs computed for comparison.")
print(f"  Raw    → {psd_raw.get_data().shape[0]} channels, {raw_for_comparison.times[-1]:.0f} s")
print(f"  Clean  → {psd_clean.get_data().shape[0]} channels, {raw_clean.times[-1]:.0f} s")

In [ ]:
# ── Helper: extract mean ± std PSD in dB ──
def psd_mean_std_db(psd_obj):
    d = 10 * np.log10(psd_obj.get_data())   # (n_ch, n_freq)
    return psd_obj.freqs, d.mean(axis=0), d.std(axis=0)

freqs_raw,   mean_raw,   std_raw   = psd_mean_std_db(psd_raw)
freqs_clean, mean_clean, std_clean = psd_mean_std_db(psd_clean)

# Shared y-axis limits for a fair visual comparison
y_min = min(mean_raw.min(), mean_clean.min()) - 5
y_max = max(mean_raw.max(), mean_clean.max()) + 5

# Frequency band shading
band_spans = [
    (0.5,  4,  '#e8d5f5', 'Delta'),
    (4,    8,  '#d5e8f5', 'Theta'),
    (8,   13,  '#d5f5e3', 'Alpha'),
    (13,  30,  '#f5f0d5', 'Beta'),
    (30,  40,  '#f5d5d5', 'Gamma'),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, freqs, mean, std, title, color in [
    (axes[0], freqs_raw,   mean_raw,   std_raw,   'RAW signal',   '#c0392b'),
    (axes[1], freqs_clean, mean_clean, std_clean, 'CLEAN signal', '#2980b9'),
]:
    # Band shading
    for f_lo, f_hi, c, label in band_spans:
        ax.axvspan(f_lo, f_hi, alpha=0.25, color=c)
        ax.text((f_lo + f_hi) / 2, y_min + 1, label,
                ha='center', va='bottom', fontsize=7, color='#666')

    # Mean line + std band
    ax.plot(freqs, mean, color=color, lw=2, label='Mean across channels')
    ax.fill_between(freqs, mean - std, mean + std,
                    alpha=0.2, color=color, label='±1 SD')

    # 50 Hz marker (line noise reference — attenuated in clean signal)
    ax.axvline(50, color='grey', lw=1, linestyle=':', alpha=0.7, label='50 Hz')

    ax.set_xlim(0.5, 80)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel('Frequency (Hz)', fontsize=11)
    ax.set_ylabel('Power (dB)', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold', color=color)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('PSD comparison — before vs after processing\nds001787', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('psd_before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'psd_before_after.png'")

---
## 10. Band power topographies

A **topography** shows how power is distributed across the scalp.  
During meditation, alpha is typically strongest over posterior (occipital) regions.

In [ ]:
# Plot topographies for each band
# (requires electrode positions to be defined in the dataset)
try:
    fig = psd.plot_topomap(
        bands={
            'Theta (4-8 Hz)': (4, 8),
            'Alpha (8-13 Hz)': (8, 13),
            'Beta (13-30 Hz)': (13, 30),
        },
        normalize=True,  # normalise relative to total power
        show=True
    )
    fig.savefig('topographies.png', dpi=150, bbox_inches='tight')
    print("Topographies saved as 'topographies.png'")
except Exception as e:
    print(f"⚠️ Topography not available: {e}")
    print("Electrode positions may not be defined in this dataset.")

---
## 11. Summary and next steps

✅ **What we did:**
- Downloaded and loaded the BIDS dataset
- Preprocessing: band-pass filter 1–40 Hz, average reference
- **Flat channel detection** — identified and dropped disconnected/dead electrodes
- **Artifact rejection with z-score** on 2-second epochs (per channel)
- Computed PSD on the clean signal using Welch's method
- Visualised band power distribution and topographies

🔜 **Possible next steps:**
- **ICA**: remove residual ocular/cardiac artifacts with Independent Component Analysis
- **Epoching**: if events are present, segment the signal into time-locked windows
- **Group comparison**: expert meditators vs novices (if the dataset includes this)
- **Time-frequency analysis**: how does alpha power evolve during the session?
- **Coherence**: how synchronised are different brain regions?